# Сетевой анализ нейронов и валидация на данных Anton-Sanchez et al.

Этот ноутбук работает с каноническим форматом линейной 3D-сети: `vertices.csv`, `edges.csv`, `spines.csv`, `matrix.npy`, `metadata.json`. Такой формат напрямую соответствует данным Anton-Sanchez et al. (`vertices`, `m`, `X`) и используется как промежуточный стандарт для MICrONS после предобработки.

Для данных статьи `metadata.json` не содержит координату сомы. В статье используется корень дерева `r`, а не mesh сомы; поэтому по умолчанию используется `root_vertex_id = 0`. Для MICrONS реальная координата сомы сохраняется отдельно в `metadata.json` как `soma_point`, а `root_vertex_id` задаёт вершину сети, от которой считаются shortest-path distances.

In [ ]:
from pathlib import Path
import importlib
import itertools
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

network_module = importlib.import_module("dendrite_analysis.network")
network_module = importlib.reload(network_module)

compute_simulation_envelopes = network_module.compute_simulation_envelopes
estimate_smooth_intensity = network_module.estimate_smooth_intensity
fit_inhomogeneous_poisson = network_module.fit_inhomogeneous_poisson
load_standard_network = network_module.load_standard_network
network_circumradius = network_module.network_circumradius
project_spines_to_graph = network_module.project_spines_to_graph
ripley_k_network = network_module.ripley_k_network
run_analysis = network_module.run_analysis

## Конфигурация

Для прямого воспроизведения статьи используем `N_SIMULATIONS = 19`, global constant-width envelope, geometrically corrected inhomogeneous network K-function и log-quadratic intensity по расстоянию до корня. Для более стабильных рабочих прогонов можно увеличить `N_SIMULATIONS` до 99.

In [ ]:
DATA_ROOT = Path("3Dnetworks_export")
OUTPUT_DIR = Path("output_anton_sanchez_validation")

ROOT_VERTEX_ID = 0
N_SIMULATIONS = 19
N_JOBS = 8
N_R_VALUES = 20
N_PERMUTATIONS = 1000
RANDOM_STATE = 42

# Быстрые тесты: поставь NETWORK_LIMIT=1 и RUN_FULL_ANALYSIS=True.
NETWORK_LIMIT = None
RUN_FULL_ANALYSIS = False
RUN_GROUP_COMPARISON = False

# Mapping basal-сетей к нейронам нужен для Fig. 6a/6c.
# В metadata после парсинга RData этой информации нет, поэтому mapping вынесен явно.
# basal_05 и basal_07 имеют малые circumradius и соответствуют исключаемым сетям Neuron 2.
BASAL_TO_NEURON = {
    "basal_01": "Neuron 1", "basal_02": "Neuron 1", "basal_03": "Neuron 1", "basal_04": "Neuron 1",
    "basal_05": "Neuron 2", "basal_06": "Neuron 2", "basal_07": "Neuron 2",
    "basal_08": "Neuron 3", "basal_09": "Neuron 3", "basal_10": "Neuron 3",
    "basal_11": "Neuron 4", "basal_12": "Neuron 4", "basal_13": "Neuron 4", "basal_14": "Neuron 4",
    "basal_15": "Neuron 5", "basal_16": "Neuron 5", "basal_17": "Neuron 5",
}

REFERENCE_VALUES = {
    "mean_apical_n_spines": 2845,
    "mean_apical_length": 2497.25,
    "mean_apical_branch_points": 20,
    "mean_basal_n_spines": 1074,
    "mean_basal_length": 951.85,
    "mean_basal_branch_points": 6,
    "fig6a_basal_by_neuron_r_max": 134.70,
    "fig6a_basal_by_neuron_p": 0.808,
    "fig6b_apical_vs_basal_r_max": 134.70,
    "fig6b_apical_vs_basal_p": 0.109,
    "fig6c_apical_vs_basal_excluding_neuron2_r_max": 165.96,
    "fig6c_apical_vs_basal_excluding_neuron2_p": 0.045,
    "basal_by_neuron_excluding_neuron2_p": 0.565,
}

ANALYSIS_KWARGS = {
    "root_vertex_id": ROOT_VERTEX_ID,
    "max_distance_to_edge": np.inf,
    "bin_size": "auto",
    "r_max": "circumradius",
    "n_r_values": N_R_VALUES,
    "n_simulations": N_SIMULATIONS,
    "n_jobs": N_JOBS,
    "k_correction": "geometric",
    "k_inhomogeneous": True,
    "envelope_type": "global_constant_width",
    "max_network_samples": 50_000,
    "max_integration_samples": 50_000,
    "random_state": RANDOM_STATE,
    "report_format": "html",
}

In [ ]:
network_dirs = []
for group in ("basal", "apical"):
    group_dir = DATA_ROOT / group
    network_dirs.extend(sorted(path for path in group_dir.glob(f"{group}_*") if path.is_dir()))

if NETWORK_LIMIT is not None:
    network_dirs = network_dirs[: int(NETWORK_LIMIT)]

print(f"Found {len(network_dirs)} networks")
network_dirs[:5]

## Визуализация всех базальных дендритов

На одном 3D-графике ниже показаны все basal-сети разными цветами. Подпись содержит имя дендрита и текущую привязку к нейрону из `BASAL_TO_NEURON`.

In [ ]:
basal_dirs = sorted(path for path in (DATA_ROOT / "basal").glob("basal_*") if path.is_dir())

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")
colors = plt.cm.tab20(np.linspace(0, 1, max(len(basal_dirs), 1)))

for color, network_dir in zip(colors, basal_dirs):
    vertices = pd.read_csv(network_dir / "vertices.csv")
    edges_path = network_dir / "edges.csv"
    spines_path = network_dir / "spines.csv"
    edges = pd.read_csv(edges_path) if edges_path.exists() else None
    spines = pd.read_csv(spines_path) if spines_path.exists() else None
    label = f"{network_dir.name} ({BASAL_TO_NEURON.get(network_dir.name, 'unknown')})"

    if edges is not None and len(edges) > 0:
        coordinates = vertices.set_index("vertex_id")
        first_edge = True
        for edge in edges.itertuples(index=False):
            source = coordinates.loc[int(edge.source)]
            target = coordinates.loc[int(edge.target)]
            ax.plot(
                [source["x"], target["x"]],
                [source["y"], target["y"]],
                [source["z"], target["z"]],
                linewidth=0.8,
                alpha=0.75,
                color=color,
                label=label if first_edge else None,
            )
            first_edge = False
    else:
        ax.scatter(
            vertices["x"],
            vertices["y"],
            vertices["z"],
            s=3,
            color=color,
            alpha=0.75,
            label=label,
        )

    if spines is not None and len(spines) > 0:
        ax.scatter(
            spines["x"],
            spines["y"],
            spines["z"],
            s=1,
            color=color,
            alpha=0.25,
        )

    center = vertices[["x", "y", "z"]].mean()
    ax.text(
        center["x"],
        center["y"],
        center["z"],
        network_dir.name,
        color=color,
        fontsize=9,
        fontweight="bold",
    )

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("Все базальные дендриты")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

plt.tight_layout()
plt.show()


basal_dirs_by_neuron = {
    neuron: sorted(path for path in basal_dirs if BASAL_TO_NEURON.get(path.name) == neuron)
    for neuron in sorted(set(BASAL_TO_NEURON.values()))
}

for neuron, neuron_basal_dirs in basal_dirs_by_neuron.items():
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection="3d")
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(neuron_basal_dirs), 1)))

    for color, network_dir in zip(colors, neuron_basal_dirs):
        vertices = pd.read_csv(network_dir / "vertices.csv")
        edges_path = network_dir / "edges.csv"
        spines_path = network_dir / "spines.csv"
        edges = pd.read_csv(edges_path) if edges_path.exists() else None
        spines = pd.read_csv(spines_path) if spines_path.exists() else None

        if edges is not None and len(edges) > 0:
            coordinates = vertices.set_index("vertex_id")
            first_edge = True
            for edge in edges.itertuples(index=False):
                source = coordinates.loc[int(edge.source)]
                target = coordinates.loc[int(edge.target)]
                ax.plot(
                    [source["x"], target["x"]],
                    [source["y"], target["y"]],
                    [source["z"], target["z"]],
                    linewidth=0.9,
                    alpha=0.8,
                    color=color,
                    label=network_dir.name if first_edge else None,
                )
                first_edge = False
        else:
            ax.scatter(
                vertices["x"],
                vertices["y"],
                vertices["z"],
                s=4,
                color=color,
                alpha=0.8,
                label=network_dir.name,
            )

        if spines is not None and len(spines) > 0:
            ax.scatter(
                spines["x"],
                spines["y"],
                spines["z"],
                s=1,
                color=color,
                alpha=0.25,
            )

        center = vertices[["x", "y", "z"]].mean()
        ax.text(
            center["x"],
            center["y"],
            center["z"],
            network_dir.name,
            color=color,
            fontsize=9,
            fontweight="bold",
        )

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(f"Базальные дендриты: {neuron}")
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), borderaxespad=0.0)

    plt.tight_layout()
    plt.show()

## Быстрая проверка структуры и аналог Table 1/Table 2

В статье Table 1/Table 2 содержат число шипиков `n`, длину сети `|L|`, плотность `n/|L|`, circumradius `R` и число branching points `#BP`. Здесь считаем тот же набор по каноническим сетям.

In [ ]:
rows = []
loaded_networks = {}

for network_dir in network_dirs:
    graph, spine_points, metadata = load_standard_network(
        network_dir,
        root_vertex_id=ROOT_VERTEX_ID,
        project_spines=False,
    )
    projected, unassigned = project_spines_to_graph(graph, spine_points, max_distance_to_edge=np.inf)
    distances = np.asarray([spine.distance_to_edge for spine in projected], dtype=float)
    group = network_dir.parent.name
    neuron_id = BASAL_TO_NEURON.get(network_dir.name, network_dir.name.replace("apical_0", "Neuron "))
    branch_points = sum(1 for _, data in graph.G.nodes(data=True) if data.get("node_type") == "branch")
    loaded_networks[network_dir.name] = {
        "group": group,
        "neuron_id": neuron_id,
        "dir": network_dir,
        "graph": graph,
        "spine_points": spine_points,
        "projected_spines": projected,
        "metadata": metadata,
    }
    rows.append({
        "network": network_dir.name,
        "group": group,
        "neuron_id": neuron_id,
        "n_spines": len(spine_points),
        "network_length": graph.total_length,
        "spine_linear_density": len(spine_points) / graph.total_length,
        "circumradius": network_circumradius(graph),
        "branch_points": branch_points,
        "root_vertex_id": graph.soma_node,
        "projection_unassigned": len(unassigned),
        "projection_distance_max": float(np.max(distances)) if len(distances) else np.nan,
        "projection_distance_median": float(np.median(distances)) if len(distances) else np.nan,
    })

structure_check = pd.DataFrame(rows).sort_values(["group", "network"]).reset_index(drop=True)
structure_check

In [ ]:
summary_by_group = structure_check.groupby("group")[[
    "n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"
]].mean()
summary_by_group

In [ ]:
table1_apical = structure_check[structure_check["group"] == "apical"].copy()
table2_basal = structure_check[structure_check["group"] == "basal"].copy()

display(table1_apical[["network", "n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"]])
display(table2_basal[["neuron_id", "network", "n_spines", "network_length", "spine_linear_density", "circumradius", "branch_points"]])

## Аналог Fig. 4: сглаженная интенсивность для первого basal arbor

В статье Fig. 4 показывает kernel-smoothed estimate интенсивности как функцию расстояния до tree root для первого basal arborization of Neuron 1.

In [ ]:
example_name = "basal_01"
example = loaded_networks[example_name]
smooth_d, smooth_lambda = estimate_smooth_intensity(example["graph"], example["projected_spines"])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(smooth_d, smooth_lambda, color="tab:red")
ax.set_title(f"Kernel-smoothed intensity: {example_name}")
ax.set_xlabel("Network distance to root")
ax.set_ylabel("Estimated intensity")
ax.grid(alpha=0.25)
fig

## Полный сетевой анализ каждой сети

Эта ячейка запускает весь reference-like pipeline: CDF-test, log-quadratic inhomogeneous Poisson model, geometrically corrected inhomogeneous network K-function и global constant-width envelope.

In [ ]:
analysis_rows = []
full_results = {}

if RUN_FULL_ANALYSIS:
    for network_dir in network_dirs:
        print(f"[Anton-Sanchez] full analysis start {network_dir.parent.name}/{network_dir.name}", flush=True)
        result = run_analysis(
            standard_network_dir=network_dir,
            output_dir=str(OUTPUT_DIR / network_dir.parent.name / network_dir.name),
            dendrite_type=network_dir.parent.name,
            timing_label=f"{network_dir.parent.name}/{network_dir.name}",
            **ANALYSIS_KWARGS,
        )
        full_results[network_dir.name] = result
        k_result = result.get("k_result")
        poisson_result = result.get("poisson_result")
        graph = result["graph"]
        analysis_rows.append({
            "network": network_dir.name,
            "group": network_dir.parent.name,
            "n_spines": len(result["projected_spines"]),
            "network_length": graph.total_length,
            "spine_linear_density": len(result["projected_spines"]) / graph.total_length,
            "circumradius": network_circumradius(graph),
            "cdf_p_value": result["intensity_cdf_test"].get("p_value"),
            "k_p_value": None if k_result is None else k_result.p_value,
            "poisson_log_likelihood": None if poisson_result is None else poisson_result.log_likelihood,
            "report_path": result.get("report_path", ""),
        })
else:
    print("RUN_FULL_ANALYSIS=False: включи True для полного анализа всех сетей.")

analysis_summary = pd.DataFrame(analysis_rows)
analysis_summary

## Аналог Fig. 5: KLI для первого basal arbor

Если полный анализ не запускался, эта ячейка отдельно считает KLI только для `basal_01`.

In [ ]:
if example_name in full_results and full_results[example_name].get("k_result") is not None:
    k_example = full_results[example_name]["k_result"]
else:
    graph = example["graph"]
    spines = example["projected_spines"]
    r_values = np.linspace(0.0, network_circumradius(graph), N_R_VALUES + 1)[1:]
    poisson = fit_inhomogeneous_poisson(graph, spines, verbose_timing=True, timing_label=example_name)
    k_example = compute_simulation_envelopes(
        graph, spines, r_values,
        n_simulations=N_SIMULATIONS,
        n_jobs=N_JOBS,
        intensity_model=poisson,
        correction="geometric",
        envelope_type="global_constant_width",
        require_inhomogeneous=True,
        random_state=RANDOM_STATE,
        timing_label=example_name,
    )

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(k_example.r_values, k_example.k_observed, label="Observed KLI", color="tab:red")
ax.plot(k_example.r_values, k_example.k_expected, label="Expected", color="black", linestyle="--")
ax.fill_between(k_example.r_values, k_example.k_lower, k_example.k_upper, color="tab:orange", alpha=0.25, label="5% global envelope")
ax.set_xlabel("Network distance")
ax.set_ylabel("KLI")
ax.set_title(f"Geometrically corrected inhomogeneous KLI: {example_name}")
ax.legend()
ax.grid(alpha=0.25)
fig

## Сравнительный анализ apical/basal и basal-by-neuron как в Fig. 6

Статья использует studentized permutation test по ранее оценённым 3D KLI-функциям. Ниже реализована приближённая воспроизводимая версия: считаем KLI каждой сети на общем диапазоне радиусов, затем сравниваем группы перестановочно. Ожидаемые p-value из статьи: basal-by-neuron на `[0, 134.70]` — `0.808`; apical-vs-basal на `[0, 134.70]` — `0.109`; после исключения Neuron 2 на `[0, 165.96]` — basal-by-neuron `0.565`, apical-vs-basal `0.045`.

In [ ]:
def compute_kli_curves(network_names, r_max, n_r_values=50):
    r_values = np.linspace(0.0, float(r_max), int(n_r_values) + 1)[1:]
    curves = {}
    for name in network_names:
        item = loaded_networks[name]
        graph = item["graph"]
        spines = item["projected_spines"]
        poisson = fit_inhomogeneous_poisson(graph, spines, verbose_timing=False)
        kli = ripley_k_network(
            graph, spines, r_values,
            intensity=poisson.fitted_intensity,
            correction="geometric",
        )
        curves[name] = kli.k_observed
    return r_values, curves


def studentized_permutation_test(curves_by_group, r_values, n_permutations=1000, random_state=42):
    rng = np.random.default_rng(random_state)
    group_names = list(curves_by_group)
    group_sizes = {group: len(curves_by_group[group]) for group in group_names}
    all_curves = []
    labels = []
    for group in group_names:
        for curve in curves_by_group[group]:
            all_curves.append(np.asarray(curve, dtype=float))
            labels.append(group)
    all_curves = np.asarray(all_curves, dtype=float)
    labels = np.asarray(labels, dtype=object)
    dr = float(r_values[1] - r_values[0]) if len(r_values) > 1 else 1.0

    def statistic(current_labels):
        group_means = []
        group_vars = []
        weights = []
        for group in group_names:
            group_curves = all_curves[current_labels == group]
            group_means.append(group_curves.mean(axis=0))
            group_vars.append(group_curves.var(axis=0, ddof=1) if len(group_curves) > 1 else np.zeros(len(r_values)))
            weights.append(len(group_curves))
        group_means = np.asarray(group_means)
        group_vars = np.asarray(group_vars)
        weights = np.asarray(weights, dtype=float)
        grand_mean = np.average(group_means, axis=0, weights=weights)
        pooled_var = np.nanmean(group_vars, axis=0)
        eps = np.nanmedian(pooled_var[pooled_var > 0]) * 1e-6 if np.any(pooled_var > 0) else 1e-12
        standardized = ((group_means - grand_mean) ** 2) / (pooled_var + eps)
        return float(np.sum(standardized) * dr)

    observed = statistic(labels)
    permuted = []
    for _ in range(int(n_permutations)):
        permuted.append(statistic(rng.permutation(labels)))
    permuted = np.asarray(permuted, dtype=float)
    p_value = float((1 + np.sum(permuted >= observed)) / (1 + len(permuted)))
    return {"p_value": p_value, "observed_statistic": observed, "permutation_statistics": permuted}


def plot_group_kli(curves_by_group, r_values, title):
    fig, ax = plt.subplots(figsize=(8, 5))
    for group, curves in curves_by_group.items():
        arr = np.asarray(curves, dtype=float)
        mean = arr.mean(axis=0)
        ax.plot(r_values, mean, label=f"{group} mean")
        for curve in arr:
            ax.plot(r_values, curve, alpha=0.18, linewidth=0.8)
    ax.plot(r_values, r_values, color="black", linestyle="--", label="Expected K=d")
    ax.set_title(title)
    ax.set_xlabel("Network distance")
    ax.set_ylabel("KLI")
    ax.legend()
    ax.grid(alpha=0.25)
    return fig

In [ ]:
comparison_results = {}

if RUN_GROUP_COMPARISON:
    basal_names = sorted([name for name, item in loaded_networks.items() if item["group"] == "basal"])
    apical_names = sorted([name for name, item in loaded_networks.items() if item["group"] == "apical"])

    # Fig. 6a: basal networks grouped by neuron, range [0, 134.70].
    r_values_134, basal_curves_134 = compute_kli_curves(basal_names, r_max=134.70)
    basal_by_neuron_134 = {
        neuron: [basal_curves_134[name] for name in basal_names if BASAL_TO_NEURON[name] == neuron]
        for neuron in sorted(set(BASAL_TO_NEURON.values()))
    }
    comparison_results["fig6a_basal_by_neuron"] = studentized_permutation_test(
        basal_by_neuron_134, r_values_134, n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE
    )
    display(plot_group_kli(basal_by_neuron_134, r_values_134, "Fig. 6a analog: basal by neuron, r≤134.70"))

    # Fig. 6b: all apical vs all basal, range [0, 134.70].
    _, apical_curves_134 = compute_kli_curves(apical_names, r_max=134.70)
    apical_vs_basal_134 = {
        "apical": [apical_curves_134[name] for name in apical_names],
        "basal": [basal_curves_134[name] for name in basal_names],
    }
    comparison_results["fig6b_apical_vs_basal"] = studentized_permutation_test(
        apical_vs_basal_134, r_values_134, n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE
    )
    display(plot_group_kli(apical_vs_basal_134, r_values_134, "Fig. 6b analog: apical vs basal, r≤134.70"))

    # Fig. 6c: exclude Neuron 2 basal networks, range [0, 165.96].
    basal_no_neuron2 = [name for name in basal_names if BASAL_TO_NEURON[name] != "Neuron 2"]
    r_values_166, basal_curves_166 = compute_kli_curves(basal_no_neuron2, r_max=165.96)
    _, apical_curves_166 = compute_kli_curves(apical_names, r_max=165.96)
    basal_by_neuron_166 = {
        neuron: [basal_curves_166[name] for name in basal_no_neuron2 if BASAL_TO_NEURON[name] == neuron]
        for neuron in sorted(set(BASAL_TO_NEURON[name] for name in basal_no_neuron2))
    }
    comparison_results["basal_by_neuron_excluding_neuron2"] = studentized_permutation_test(
        basal_by_neuron_166, r_values_166, n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE
    )
    apical_vs_basal_166 = {
        "apical": [apical_curves_166[name] for name in apical_names],
        "basal_without_neuron2": [basal_curves_166[name] for name in basal_no_neuron2],
    }
    comparison_results["fig6c_apical_vs_basal_excluding_neuron2"] = studentized_permutation_test(
        apical_vs_basal_166, r_values_166, n_permutations=N_PERMUTATIONS, random_state=RANDOM_STATE
    )
    display(plot_group_kli(apical_vs_basal_166, r_values_166, "Fig. 6c analog: apical vs basal without Neuron 2, r≤165.96"))
else:
    print("RUN_GROUP_COMPARISON=False: включи True для studentized permutation comparisons.")

REFERENCE_P_VALUES = {
    "fig6a_basal_by_neuron": 0.808,
    "fig6b_apical_vs_basal": 0.109,
    "basal_by_neuron_excluding_neuron2": 0.565,
    "fig6c_apical_vs_basal_excluding_neuron2": 0.045,
}

pd.DataFrame([
    {
        "comparison": name,
        "our_p_value": result["p_value"],
        "reference_p_value": REFERENCE_P_VALUES.get(name),
        "observed_statistic": result["observed_statistic"],
    }
    for name, result in comparison_results.items()
])